<a href="https://colab.research.google.com/github/Zattyla/RVC-UVR-Pipeline/blob/main/AIStudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sobre a Colab: UVR5 + RVC v2 Pipeline
> **A solução "tudo-em-um" para criar AI Covers de alta qualidade.**

Este notebook integra o poder do **UVR (Ultimate Vocal Remover)** para separação de áudio e o **RVC (Retrieval-based Voice Conversion)** para transformação de voz.

### 🛠️ Como funciona?
1.  **UVR Separation:** Filtra a música original, separando o Instrumental do Vocal.
2.  **Vocal Cleaner:** Remove eco e reverberação do vocal isolado.
3.  **RVC Conversion:** Transforma a voz original na voz do modelo de IA escolhido.
4.  **Auto-Mixing:** Junta a nova voz com o instrumental original automaticamente.

---
⚠️ **Aviso:** Este projeto é para fins educacionais e criativos. Respeite os direitos autorais e use modelos de voz de forma ética.

In [ ]:
# @title 🧹 Limpar Logs e Mensagens
from IPython.display import clear_output

# @markdown Clique aqui para esconder toda a sujeira das instalações anteriores.
clear_output()
print("✅ Interface limpa! O ambiente está pronto para uso.")

In [ ]:
# @title 1. GPU Check & Environment Setup
import os
import subprocess

# Verifica a GPU
try:
    gpu_info = !nvidia-smi
    if "nvidia-smi" in gpu_info[0] or "failed" in gpu_info[0]:
        print("❌ GPU não detectada. Vá em 'Ambiente de Execução' -> 'Alterar tipo...' e selecione T4 GPU.")
    else:
        print("✅ GPU Detectada!")
        print(gpu_info[0])
except:
    print("❌ Erro ao executar nvidia-smi. Certifique-se de que a GPU está ativada.")

# Definição de caminhos do sistema (Paths)
BASE_DIR = "/content"
DRIVE_DIR = "/content/drive/MyDrive/RVC_UVR_Colab"

In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Criar pasta principal no Drive se não existir
if not os.path.exists(DRIVE_DIR):
    os.makedirs(DRIVE_DIR)
    print(f"✨ Pasta criada no Drive: {DRIVE_DIR}")
else:
    print(f"📂 Pasta do projeto já existe no Drive.")

In [ ]:
# @title 📦 Backup de Resultados para o Drive
import shutil
from datetime import datetime

# @markdown Isso vai compactar a pasta 'outputs' e salvar no seu Drive.
DATA_ATUAL = datetime.now().strftime("%Y-%m-%d_%H-%M")
NOME_BACKUP = f"Meus_Covers_{DATA_ATUAL}"
CAMINHO_DESTINO = f"{DRIVE_DIR}/Backups/{NOME_BACKUP}"

if not os.path.exists(f"{DRIVE_DIR}/Backups"):
    os.makedirs(f"{DRIVE_DIR}/Backups")

print(f"⏳ Criando backup em {NOME_BACKUP}.zip...")
shutil.make_archive(CAMINHO_DESTINO, 'zip', "/content/outputs")

print(f"✅ Backup concluído! Salvo em: {DRIVE_DIR}/Backups")

In [ ]:
# @title 3. Install Dependencies (Fixed)
print("⏳ Instalando dependências corrigidas...")

# 1. Atualiza pacotes do sistema
!apt-get update -y && apt-get install -y ffmpeg aria2

# 2. Instala bibliotecas de áudio e interface
!pip install -q pydub librosa gradio==3.48.0

# 3. Instala o áudio-separator (Essencial para o UVR)
!pip install -q audio-separator[gpu]

# 4. Instala Faiss e Fairseq (Essenciais para o RVC)
# Removemos a versão fixa para o pip escolher a melhor disponível
!pip install -q faiss-cpu fairseq pyworld

# 5. Garante que o Tocha está com suporte a CUDA (GPU)
!pip install -q torch torchvision torchaudio

print("✅ Dependências instaladas com sucesso!")

In [ ]:
# @title 4. Download Core & Create Folders
%cd /content

# Clonar o Applio (Motor RVC mais estável atualmente)
if not os.path.exists("Applio"):
    !git clone https://github.com/IAHispano/Applio.git
    %cd Applio
    !pip install -r requirements.txt
else:
    print("🚀 Applio já clonado.")

# Criar pastas organizadas para o seu fluxo
folders = [
    "inputs",           # Onde você sobe sua música
    "outputs/uvr",      # Resultado da separação (vocal/inst)
    "outputs/rvc",      # Resultado da conversão de voz
    "models/rvc",       # Seus modelos .pth e .index
    "models/uvr"        # Modelos de separação MDX/VR
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("✅ Estrutura de pastas pronta.")

In [ ]:
# @title 5. Download Separation Models (UVR)
# Link para o modelo Kim_Vocal_2 (Um dos melhores para isolar vocais limpos)
UVR_MODEL_URL = "https://huggingface.co/MohamedH97/UVR-Model-Parent-Directory/resolve/main/MDX-Net/Kim_Vocal_2.onnx"
UVR_MODEL_PATH = "/content/models/uvr/Kim_Vocal_2.onnx"

if not os.path.exists(UVR_MODEL_PATH):
    print("⏬ Baixando modelo de separação UVR...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {UVR_MODEL_URL} -d /content/models/uvr -o Kim_Vocal_2.onnx
    print("✅ Modelo UVR pronto.")
else:
    print("✔ Modelo UVR já existe.")

In [ ]:
# @title 6. Voice Library (Download RVC Model)
import zipfile
import shutil

# @markdown Cole o link direto do modelo (HuggingFace/Drive/Pixeldrain)
MODEL_URL = "" # @param {type:"string"}
MODEL_NAME = "Meu_Modelo_Voz" # @param {type:"string"}

def download_model(url, name):
    if not url:
        print("⚠ Por favor, insira o link de um modelo.")
        return

    model_dir = f"/content/Applio/logs/{name}"
    os.makedirs(model_dir, exist_ok=True)

    print(f"⏳ Baixando modelo: {name}...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d {model_dir} -o "model.zip"

    # Extrair se for zip
    zip_path = os.path.join(model_dir, "model.zip")
    if zipfile.is_zipfile(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(model_dir)
        os.remove(zip_path)
        print(f"✅ Modelo {name} extraído e pronto!")
    else:
        print("❌ O arquivo baixado não é um ZIP válido ou o link está quebrado.")

download_model(MODEL_URL, MODEL_NAME)

In [ ]:
# @title 7. Download Utilities (RMVPE & Base Models)
print("⏳ Baixando utilitários essenciais para detecção de voz...")

# Download RMVPE (O 'ouvido' do RVC)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt" -d /content/Applio/rvc/models/predictors -o rmvpe.pt

# Download de modelos base (Pretraineds) para evitar erros de inicialização
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/Pretraineds/f0G40k.pth" -d /content/Applio/rvc/models/pretraineds -o f0G40k.pth

print("✅ Utilitários prontos para uso.")

In [ ]:
# @title 8. UVR Separation (Vocal vs Instrumental)
import os

# @markdown Nome do arquivo que você subiu na pasta 'inputs' (ex: musica.mp3)
AUDIO_INPUT = "musica.mp3" # @param {type:"string"}
input_path = f"/content/inputs/{AUDIO_INPUT}"
output_path = "/content/outputs/uvr"

if os.path.exists(input_path):
    print(f"🎵 Iniciando separação de: {AUDIO_INPUT}...")

    # Usando o audio-separator via CLI para economizar RAM
    !audio-separator "{input_path}" \
        --model_filename Kim_Vocal_2.onnx \
        --output_dir "{output_path}" \
        --output_format WAV \
        --normalization 0.9

    print(f"✅ Separação concluída! Verifique a pasta: {output_path}")
else:
    print(f"❌ Erro: O arquivo {AUDIO_INPUT} não foi encontrado na pasta /content/inputs/")

In [ ]:
# @title 9. Vocal Enhancer (Limpeza de Reverb e Echo)
# @markdown Isso remove o "eco" do vocal original para a IA soar mais natural.
CLEAN_MODEL_URL = "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/uvr5_weights/onnx_dereverb_By_Spleeter.onnx"
CLEAN_MODEL_PATH = "/content/models/uvr/dereverb.onnx"

# Baixa o modelo de de-reverb se não existir
if not os.path.exists(CLEAN_MODEL_PATH):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {CLEAN_MODEL_URL} -d /content/models/uvr -o dereverb.onnx

# @markdown Digite o nome do arquivo de vocal gerado na célula anterior (ex: musica_(Vocals)_Kim_Vocal_2.wav)
VOCAL_INPUT = "" # @param {type:"string"}
vocal_path = f"/content/outputs/uvr/{VOCAL_INPUT}"

if os.path.exists(vocal_path):
    print(f"🧹 Limpando eco de: {VOCAL_INPUT}...")
    !audio-separator "{vocal_path}" \
        --model_filename dereverb.onnx \
        --output_dir "/content/outputs/uvr" \
        --output_format WAV
    print("✅ Vocal limpo e pronto para a conversão!")
else:
    print("❌ Arquivo de vocal não encontrado.")

In [ ]:
# @title 10 & 11. RVC Voice Conversion
import sys
%cd /content/Applio

# @markdown ### Configurações da Conversão
# @markdown Nome do Vocal Limpo (gerado na célula 9)
INPUT_VOCAL = "" # @param {type:"string"}
# @markdown Nome do Modelo de Voz (mesmo nome usado na Célula 6)
MODEL_NAME = "Meu_Modelo_Voz" # @param {type:"string"}
# @markdown Ajuste de Tom (Pitch): 0 mantém original, 12 sobe uma oitava (masculino -> feminino), -12 desce uma oitava.
PITCH = 0 # @param {type:"slider", min:-24, max:24, step:1}
# @markdown Algoritmo de Pitch: 'rmvpe' é o melhor para quase tudo.
METHOD = "rmvpe" # @param ["rmvpe", "mangio-crepe"]

input_path = f"/content/outputs/uvr/{INPUT_VOCAL}"
output_path = f"/content/outputs/rvc/{INPUT_VOCAL}_IA.wav"

if os.path.exists(input_path):
    print(f"🎤 Convertendo voz para {MODEL_NAME}...")

    # Comando CLI do Applio para inferência
    !python infer.py \
        --input_path "{input_path}" \
        --output_path "{output_path}" \
        --model_name "{MODEL_NAME}" \
        --transpose {PITCH} \
        --f0_method "{METHOD}" \
        --index_rate 0.75 \
        --volume_envelope 1 \
        --protect 0.33

    print(f"🎉 SUCESSO! Sua música com IA está em: {output_path}")
else:
    print("❌ Erro: Vocal de entrada não encontrado.")

In [ ]:
# @title 12. Audio Mixing (Final Master)
from pydub import AudioSegment
import os

# @markdown ### Ajustes de Volume (em dB)
VOCAL_VOLUME = 0 # @param {type:"slider", min:-20, max:10, step:1}
INSTRUMENTAL_VOLUME = 0 # @param {type:"slider", min:-20, max:10, step:1}

# @markdown Nome do Instrumental (gerado no Passo 8)
INST_FILE = "" # @param {type:"string"}
# @markdown Nome do Vocal RVC (gerado no Passo 11)
RVC_VOCAL_FILE = "" # @param {type:"string"}

inst_path = f"/content/outputs/uvr/{INST_FILE}"
rvc_path = f"/content/outputs/rvc/{RVC_VOCAL_FILE}"
final_output = f"/content/outputs/FINAL_COVER_{RVC_VOCAL_FILE}.mp3"

if os.path.exists(inst_path) and os.path.exists(rvc_path):
    print("🎚️ Mixando faixas...")
    vocal = AudioSegment.from_file(rvc_path) + VOCAL_VOLUME
    instrumental = AudioSegment.from_file(inst_path) + INSTRUMENTAL_VOLUME

    # Sobrepor vocal no instrumental
    combined = instrumental.overlay(vocal)

    # Exportar
    combined.export(final_output, format="mp3")
    print(f"✨ COVER FINALIZADO! Baixe aqui: {final_output}")
else:
    print("❌ Erro: Verifique se os nomes dos arquivos de Instrumental e Vocal estão corretos.")

In [ ]:
# @title 14 & 15. Launch Full App (Interface Gradio)
import gradio as gr

def run_pipeline(audio_input, model_url, pitch):
    # Aqui dentro colocaríamos as chamadas de função simplificadas
    # das células 6, 8, 9, 11 e 12.
    return "Processamento iniciado... Verifique as pastas de output!"

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🎤 AI Cover Maker (UVR5 + RVC v2)")

    with gr.Row():
        with gr.Column():
            input_audio = gr.Audio(label="Suba sua música original", type="filepath")
            model_link = gr.Textbox(label="Link do Modelo RVC (ZIP)", placeholder="https://huggingface.co/...")
            pitch_shift = gr.Slider(minimum=-12, maximum=12, value=0, label="Ajuste de Tom (Pitch)")
            btn = gr.Button("🚀 Gerar IA Cover", variant="primary")

        with gr.Column():
            output_info = gr.Textbox(label="Status")
            final_audio = gr.Audio(label="Seu Cover Final")

    gr.Markdown("---")
    gr.Markdown("### 🛠️ Estrutura por trás do capô:")
    gr.Markdown("1. Separação UVR (Kim_Vocal_2) | 2. De-reverb | 3. Inferência RVC | 4. Mixagem Pydub")

app.launch(share=True, debug=True)

## 🛠️ Solução de Problemas (Troubleshooting)

Se algo der errado, não entre em pânico! Aqui estão as soluções para os erros mais comuns:

### 1. 🛑 Erro de "Out of Memory" (OOM) na GPU
Isso acontece quando o modelo de áudio é muito pesado para a placa de vídeo.
* **Solução:** Vá em `Ambiente de Execução` -> `Reiniciar sessão`. Certifique-se de não rodar o UVR e o RVC ao mesmo tempo. Rode a célula de "Limpar Logs" para liberar memória.

### 2. 🔇 O áudio final está mudo ou com chiado
Geralmente é causado por um volume de entrada muito alto que causou "clipping".
* **Solução:** No **Passo 12 (Mixing)**, abaixe o `VOCAL_VOLUME` para -3 ou -6 dB.

### 3. 📁 Arquivo não encontrado (FileNotFoundError)
O Python é sensível a espaços e caracteres especiais (acentos, cedilha).
* **Solução:** Renomeie sua música na pasta `inputs` para algo simples, como `musica.mp3`, antes de rodar as células de processamento.

### 4. 🎤 A voz da IA está soando robótica ou "metálica"
Isso acontece se o vocal original tiver muito eco ou se o modelo RVC for de baixa qualidade.
* **Solução:** * Certifique-se de rodar o **Passo 9 (Vocal Enhancer)**.
    * Tente mudar o `METHOD` de `rmvpe` para `mangio-crepe` no Passo 11.
    * Aumente o `index_rate` para 0.8 ou 0.9.

### 5. ⏳ O download do modelo falhou
Links do Google Drive costumam expirar ou exigir permissão.
* **Solução:** Use links diretos do **HuggingFace** ou **Pixeldrain**. Se o arquivo for um `.zip`, verifique se ele realmente contém os arquivos `.pth` e `.index`.

---
🔧 **Dica:** Se o Colab travar completamente, use `Ambiente de Execução` -> `Desconectar e excluir ambiente de execução` para começar do zero com uma GPU limpa.